# BIDS, pybids & dcm2niix: Data Management

Query BIDS datasets programmatically, convert DICOM to NIfTI, and understand the data organization standard that every modern neuroimaging lab uses.

## Prerequisites

- **Python 3.8+** with pybids installed
- **dcm2niix** installed system-wide for DICOM conversion (optional but recommended)
  - On Ubuntu/Debian: `sudo apt install dcm2niix`
  - On macOS: `brew install dcm2niix`
- **heudiconv** for automated BIDS conversion (optional)
- **DataLad** for downloading OpenNeuro datasets
- Basic understanding of the BIDS standard: https://bids-specification.readthedocs.io/

## Setup

1. Install the Python dependencies (cell below)
2. Download the required datasets via DataLad (cell below)

In [ ]:
# Install required packages
!pip install -q pybids pandas datalad

In [ ]:
# Download datasets via DataLad (run once, then comment out)
# !datalad install https://github.com/OpenNeuroDatasets/ds000114
# !datalad install https://github.com/OpenNeuroDatasets/ds000228

In [ ]:
from bids import BIDSLayout
import os, subprocess, pandas as pd

# Portable paths -- relative to notebook directory
DS114 = 'ds000114'
DS228 = 'ds000228'

# Load BIDS dataset -- pybids builds a full database of everything in the dataset
print('Indexing ds000114...')
layout = BIDSLayout(DS114, validate=False)
print(layout)

In [ ]:
# Query the dataset like a database
print('=== Subjects ===')
print(layout.get_subjects())

print('\n=== Tasks ===')
print(layout.get_tasks())

print('\n=== All T1w files ===')
t1s = layout.get(suffix='T1w', extension='.nii.gz')
for f in t1s: print(' ', f.path)

print('\n=== fMRI files for sub-01 ===')
funcs = layout.get(subject='01', suffix='bold', session='test', extension='.nii.gz')
for f in funcs: print(f'  task={f.get_metadata().get("TaskName","?")} | {f.path}')

In [ ]:
# Get metadata (TR, SliceTiming, etc.) for any file
func = funcs[0]
meta = func.get_metadata()
print('=== fMRI Metadata ===')
for k, v in meta.items(): print(f'  {k}: {v}')

In [ ]:
# Build a subjects x confounds table
participants = pd.read_csv(f'{DS228}/participants.tsv', sep='\t')
layout228 = BIDSLayout(DS228, validate=False)

print('ds000228 subjects x metadata:')
print(participants[['participant_id', 'Age', 'Gender', 'Child_Adult']].head(10))
print(f'\nTotal subjects: {len(participants)}')
print(f'Children: {(participants.Child_Adult == "child").sum()}')
print(f'Adults:   {(participants.Child_Adult == "adult").sum()}')

In [ ]:
# dcm2niix demo -- shows how to convert DICOM scanner output
result = subprocess.run(['dcm2niix', '--version'], capture_output=True, text=True)
print('dcm2niix version:', result.stdout.strip() or result.stderr.strip())
print()
print('To convert a DICOM folder to BIDS-ready NIfTI:')
print('  dcm2niix -b y -z y -f %p_%s -o /output/dir /path/to/DICOM/')
print()
print('  -b y    -> generate BIDS sidecar .json')
print('  -z y    -> gzip output')
print('  -f %p_%s -> filename = protocol_series')
print()
print('For automated BIDS conversion from a full study, use heudiconv:')
result2 = subprocess.run(['heudiconv', '--version'], capture_output=True, text=True)
print('  heudiconv', result2.stdout.strip() or result2.stderr.strip())

## References

- Gorgolewski, K. J., Auer, T., Calhoun, V. D., et al. (2016). The brain imaging data structure, a format for organizing and describing outputs of neuroimaging experiments. *Scientific Data*, 3, 160044. https://doi.org/10.1038/sdata.2016.44
- Yarkoni, T., Markiewicz, C. J., de la Vega, A., et al. (2019). PyBIDS: Python tools for BIDS datasets. *Journal of Open Source Software*, 4(40), 1294. https://doi.org/10.21105/joss.01294
- Li, X., Morgan, P. S., Ashburner, J., Smith, J., & Rorden, C. (2016). The first step for neuroimaging data analysis: DICOM to NIfTI conversion. *Journal of Neuroscience Methods*, 264, 47-56. (dcm2niix)
- Halchenko, Y. O., et al. (2024). HeuDiConv -- flexible DICOM conversion into structured directory layouts. *Journal of Open Source Software*, 9(99), 5839. https://doi.org/10.21105/joss.05839
- Richardson, H., Lisandrelli, G., Riobueno-Naylor, A., & Saxe, R. (2018). Development of the social brain from age three to twelve years. *Nature Communications*, 9(1), 1027. (ds000228)